In [0]:
# Read from Unity Catalog Silver Delta table
df_silver = spark.read.table("fraud_analytics.silver.credit_card_fraud")

print(f"Silver row count: {df_silver.count()}")

In [0]:
from pyspark.sql.functions import col, sum, count, round, when

# Fraud rate by location
df_gold_location = df_silver \
    .groupBy("location") \
    .agg(
        count("transaction_id").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_count"),
        round(sum("amount"), 2).alias("total_amount"),
        round(sum(when(col("is_fraud") == True, col("amount")).otherwise(0)), 2).alias("fraud_amount")
    ) \
    .withColumn("fraud_rate_pct", 
        round((col("fraud_count") / col("total_transactions")) * 100, 2)) \
    .orderBy(col("fraud_count").desc())

df_gold_location.show(10)

In [0]:
from pyspark.sql.functions import to_date

# Highest fraud count by day
df_gold_daily = df_silver \
    .withColumn("transaction_date_only", to_date(col("transaction_date"))) \
    .groupBy("transaction_date_only") \
    .agg(
        count("transaction_id").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_count"),
        round(sum(when(col("is_fraud") == True, col("amount")).otherwise(0)), 2).alias("fraud_amount")
    ) \
    .orderBy(col("fraud_count").desc())

df_gold_daily.show(10)

In [0]:
# Merchant with most fraud
df_gold_merchant = df_silver \
    .groupBy("merchant_id") \
    .agg(
        count("transaction_id").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_count"),
        round(sum(when(col("is_fraud") == True, col("amount")).otherwise(0)), 2).alias("fraud_amount")
    ) \
    .withColumn("fraud_rate_pct",
        round((col("fraud_count") / col("total_transactions")) * 100, 2)) \
    .orderBy(col("fraud_count").desc())

df_gold_merchant.show(10)

In [0]:
# Fraud by transaction type - purchase vs refund
df_gold_txn_type = df_silver \
    .groupBy("transaction_type") \
    .agg(
        count("transaction_id").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_count"),
        round(sum(when(col("is_fraud") == True, col("amount")).otherwise(0)), 2).alias("fraud_amount")
    ) \
    .withColumn("fraud_rate_pct",
        round((col("fraud_count") / col("total_transactions")) * 100, 2)) \
    .orderBy(col("fraud_rate_pct").desc())

df_gold_txn_type.show()

In [0]:
# Monthly fraud trend
df_gold_monthly = df_silver \
    .groupBy("transaction_year", "transaction_month") \
    .agg(
        count("transaction_id").alias("total_transactions"),
        sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_count"),
        round(sum(when(col("is_fraud") == True, col("amount")).otherwise(0)), 2).alias("fraud_amount")
    ) \
    .withColumn("fraud_rate_pct",
        round((col("fraud_count") / col("total_transactions")) * 100, 2)) \
    .orderBy("transaction_year", "transaction_month")

df_gold_monthly.show(15)

In [0]:
from pyspark.sql.functions import current_timestamp

# Add aggregated_at timestamp to each gold table
df_gold_location = df_gold_location.withColumn("aggregated_at", current_timestamp())
df_gold_merchant = df_gold_merchant.withColumn("aggregated_at", current_timestamp())
df_gold_daily = df_gold_daily.withColumn("aggregated_at", current_timestamp())
df_gold_txn_type = df_gold_txn_type.withColumn("aggregated_at", current_timestamp())
df_gold_monthly = df_gold_monthly.withColumn("aggregated_at", current_timestamp())

# Write to Unity Catalog gold schema
df_gold_location.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud_analytics.gold.fraud_by_location")

df_gold_merchant.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud_analytics.gold.fraud_by_merchant")

df_gold_daily.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud_analytics.gold.fraud_by_day")

df_gold_txn_type.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud_analytics.gold.fraud_by_transaction_type")

df_gold_monthly.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud_analytics.gold.fraud_by_month")

print("All Gold tables written successfully!")

In [0]:
# Verify all gold tables
spark.sql("SHOW TABLES IN fraud_analytics.gold").show()

In [0]:
# What city has the highest fraud rate?
spark.sql("""
    SELECT location, total_transactions, fraud_count, 
           fraud_amount, fraud_rate_pct
    FROM fraud_analytics.gold.fraud_by_location
    ORDER BY fraud_rate_pct DESC
    LIMIT 5
""").show()

In [0]:
# Purchase vs Refund fraud comparison
spark.sql("""
    SELECT * FROM fraud_analytics.gold.fraud_by_transaction_type
""").show()